# **08. SHAP Explainability**

In [1]:
# Import Libraries

import sys
sys.path.append("..")
from src.utilis import *

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import pickle
import joblib
import shap

sns.set_style('darkgrid')


## **1. Loading Preprocessed Data & Best Tuning Model**

In [2]:
display_md("**Loading Preprocessed data & best tuning model...**")

# Define the path to the preprocessed data and best tuning model
parent_path = Path.cwd().parent
processed_data_path = parent_path.joinpath('models', 'best_model.joblib')
tuned_model_path = parent_path.joinpath("models", "tuned_model.joblib")

# Load the necessary data and model for SHAP explainability
try:
    with open(tuned_model_path, 'rb') as f:
        tuned_model = joblib.load(f)

    with open(processed_data_path, 'rb') as f:
        processed_data = joblib.load(f)

    X_train, X_test = processed_data['X_train'], processed_data['X_test']
    y_train, y_test = processed_data['y_train'], processed_data['y_test']
    tuned_model_pipeline = tuned_model

except Exception as e:
    display_md(f"Error loading data or model: {e}") 


**Loading Preprocessed data & best tuning model...**

## **2. Preparing Data & Model For SHAP Explainability**

In [15]:
display_md("**============= Preparing Data and Model for SHAP Explainability =============**")

# Retrain the champion model with the best hyperparameters on the entire training dataset
tuned_model_pipeline.fit(X_train, y_train)

# Model Explainability using SHAP
print("\n----- Generating SHAP Explainations -----")

# Step 1: Extract fitted preprocessed data and model
tuned_pipeline = tuned_model_pipeline.calibrated_classifiers_[0].estimator
preprocessor = tuned_pipeline.named_steps['preprocessor']
tuned_model = tuned_pipeline.named_steps['model']

# Step 2: Transform X_train and X_test
X_train_transformed = preprocessor.transform(X_train)
X_test_transformed = preprocessor.transform(X_test)

# Step 3: Get the transformed feature names
transformed_feature_names = preprocessor.get_feature_names_out()
selected_features = [
    name.replace("numeric__", "").replace("cat__", "")
    for name in transformed_feature_names.tolist()
]

# Step 4: Create the SHAP TreeExplainer
explainer = shap.TreeExplainer(
    tuned_model
    #X_train_transformed,

)

# Step 5: Calculate SHAP values for the test set
shap_value = explainer.shap_values(X_test_transformed)
shap_obj = explainer(X_test_transformed)

# --- Plot 1: SHAP Summary Plot ---
# Step 6: Visualize the summary plot
print("\n1. Global Feature Importance (Summary Plot)")
plt.figure(figsize=(10, 6))
shap.summary_plot(
    shap_value,
    X_test_transformed,
    feature_names = selected_features,
    show=False
)
plt.title("SHAP Summary Plot (Global Impact)", fontsize=14, fontweight='bold')
plt.tight_layout()
#plt.savefig("../images/shap_summary_plot.png", bbox_inches='tight')
plt.show()

# ---- Plot 2: SHAP Waterfall Plot ----
# Step 7: Waterfall plot : explain one contact/campaign observation
print("\n2. Local Explanation: Why did the model predict this specific subscribed / not subscribed?")
plt.figure(figsize=(10, 6))

shap_values = explainer(X_test_transformed)
shap_exp = shap.Explanation(
    values = shap_values.values[0],
    base_values= shap_values.base_values[0],
    data = X_test_transformed[0].toarray().flatten()
    if hasattr(X_test_transformed[0], "toarray")
    else X_test_transformed[0],
    feature_names=selected_features
)
shap.plots.waterfall(
    shap_exp,
    #max_display=10,
    show=False
)
plt.title("SHAP Waterfall Plot (Single House Prediction)", fontsize=14, fontweight='bold')
#plt.savefig("../images/shap_waterfall_plot.png", bbox_inches='tight')
plt.tight_layout()
plt.show()

# --- Plot 3: SHAP Dependence Plot ---
# Step 8: Dependence Plot
print("\n3. Feature Dependence Plot: Poutcome vs. Price Impact")
plt.figure(figsize=(10, 6))
shap.dependence_plot(
    "poutcome",
    shap_value,
    X_test_transformed,
    feature_names=selected_features,
    show=False
)
plt.title("SHAP Dependence Plot (Poutcome)", fontsize=14, fontweight='bold')
#plt.savefig("../images/shap_dependence_plot.png", bbox_inches='tight')
plt.tight_layout()
plt.show()



**============= Preparing Data and Model for SHAP Explainability =============**


----- Generating SHAP Explainations -----


ValueError: could not convert string to float: '[1.1699889E-1]'

In [17]:
print("Transformed data type:", type(X_test_transformed))
print("Transformed data dtype:",
      getattr(X_test_transformed, "dtype", "No dtype"))

print("Sample transformed values:")
#print(X_test_transformed[:5])
from scipy.sparse import issparse

print("Is sparse:", issparse(X_test_transformed))

Transformed data type: <class 'numpy.ndarray'>
Transformed data dtype: float64
Sample transformed values:
Is sparse: False


In [18]:
print("Model type:", type(tuned_model))

print("Model parameters:")
print(tuned_model.get_params())

print("\nTrain transformed:")
print(X_train_transformed.dtype)

print("\nTest transformed:")
print(X_test_transformed.dtype)

print("\nNumeric check:")
print(np.isfinite(X_test_transformed).all())

print("\nFeature count:")
print(len(transformed_feature_names))

print("\nTransformed shape:")
print(X_test_transformed.shape)


Model type: <class 'xgboost.sklearn.XGBClassifier'>
Model parameters:
{'objective': 'binary:logistic', 'base_score': None, 'booster': None, 'callbacks': None, 'colsample_bylevel': None, 'colsample_bynode': None, 'colsample_bytree': 1.0, 'device': None, 'early_stopping_rounds': None, 'enable_categorical': False, 'eval_metric': 'logloss', 'feature_types': None, 'feature_weights': None, 'gamma': 0.5, 'grow_policy': None, 'importance_type': None, 'interaction_constraints': None, 'learning_rate': None, 'max_bin': None, 'max_cat_threshold': None, 'max_cat_to_onehot': None, 'max_delta_step': None, 'max_depth': 5, 'max_leaves': None, 'min_child_weight': 1, 'missing': nan, 'monotone_constraints': None, 'multi_strategy': None, 'n_estimators': None, 'n_jobs': None, 'num_parallel_tree': None, 'random_state': 42, 'reg_alpha': None, 'reg_lambda': None, 'sampling_method': None, 'scale_pos_weight': None, 'subsample': 1.0, 'tree_method': None, 'validate_parameters': None, 'verbosity': None, 'use_label_

In [ ]:
import shap
import xgboost
import sklearn

print("SHAP version:", shap.__version__)
print("XGBoost version:", xgboost.__version__)
print("Sklearn version:", sklearn.__version__)

In [19]:
explainer = shap.TreeExplainer(tuned_model)
print("TreeExplainer initialized successfully")

ValueError: could not convert string to float: '[1.1699889E-1]'